# Create fake GCNs for our events

In [1]:
import json
import random
import pandas as pd
import string
from pathlib import Path
import copy

import astropy.units as u

### Sample Data

In [2]:
# sample data

ex = {
    "alert_type": "PRELIMINARY",
    "time_created": "2018-11-01T22:34:49Z",
    "superevent_id": "MS181101ab",
    "urls": {
        "gracedb": "https://example.org/superevents/MS181101ab/view/"
    },
    "event": {
        "time": "2018-11-01T22:22:46.654Z",
        # "far": 9.11069936486e-14,
        "far": 9.0e-14,
        "significant": True,
        "instruments": [
            "H1",
            "L1",
            "V1"
        ],
        "group": "CBC",
        "pipeline": "gstlal",
        "search": "MDC",
        "properties": {
            "HasNS": 0.95,
            "HasRemnant": 0.91,
            "HasMassGap": 0.01
        },
        "classification": {
            "BNS": 0.95,
            "NSBH": 0.01,
            "BBH": 0.03,
            "Terrestrial": 0.01
        },
        "duration": None,
        "central_frequency": None,
        "skymap": "U0lNUExFICA9ICAgICAgICAgICAgICAgICAgICBUIC8gY29uZm..."
    },
    "external_coinc": None
}

### fold in event info

In [3]:
events = pd.read_csv("./ctao_event_times.csv", parse_dates=["timestamp"])
info = pd.read_csv("../O5_event_info.csv", converters={"ifos": eval})

In [27]:
def get_event_info(info_df: pd.DataFrame, event_id: int):
    i = info_df.iloc[event_id]

    return {
        "distance": (i.dist * u.kpc).to(u.Mpc),
        "z": i.z,
        "ifos": i.ifos,
    }

get_event_info(info, 1856)


{'distance': <Quantity 140. Mpc>, 'z': np.float64(0.031), 'ifos': ['H1', 'K1']}

In [75]:
def create_gcn(row: pd.Series, info_df: pd.DataFrame, sample_data: dict, output_dir: Path | str | None = None):
    
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    info = get_event_info(info_df, row.file_id)
    ifos = info["ifos"]
    event_time = row.timestamp

    # alert time is 10 seconds later
    alert_time = event_time + pd.Timedelta(seconds=10)
    
    # copy sample data
    gcn = copy.deepcopy(sample_data)
    
    gcn["superevent_id"] = row.superevent_id
    gcn["time_created"] = alert_time.isoformat() + "Z"
    gcn["urls"]["gracedb"] = f"https://example.org/superevents/{row.superevent_id}/view/"
    gcn["event"]["time"] = event_time.isoformat() + "Z"
    gcn["event"]["instruments"] = ifos
    
    if output_dir is not None:
        output_file = output_dir / f"{row.superevent_id}_GCN.json"
        with open(output_file, "w") as f:
            json.dump(gcn, f, indent=4)
    
    else:
        return gcn


In [80]:
_ = events.apply(
    lambda row: create_gcn(row, info, ex, "../GCNs"),
    axis=1
)

## Create a metadata table

In [43]:
df = events.copy()

df["model_filepath"] = "gammapy_models/" + df.superevent_id + "_" + df.file_name.str.split(".").str[0] + "_gammapy.fits"

# rename file_id to model_id
df.rename(columns={"file_id": "model_id", "timestamp": "timestamp_utc"}, inplace=True)

# drop file_name column
df.drop(columns=["file_name", "event_id"], inplace=True)

# add gcn filepath
df["gcn_filepath"] = "GCNs/" + df.superevent_id + "_GCN.json"

df["distance_mpc"] = df.apply(lambda row: get_event_info(info, row.model_id)["distance"].value, axis=1)
df["z"] = df.apply(lambda row: get_event_info(info, row.model_id)["z"], axis=1)

# empty list for pointings
df["pointings"] = df.apply(lambda row: [], axis=1)  

# create event_id from 0
df["sdc_event_id"] = range(len(df))

# arrange columns
df = df[["sdc_event_id", "superevent_id", "model_id", "model_filepath", "gcn_filepath", "timestamp_utc", "distance_mpc", "z", "pointings"]]


In [44]:
df.to_csv("../CTAO-SDC-GW-metadata.csv", index=False)

In [45]:
df

,sdc_event_id,superevent_id,model_id,model_filepath,gcn_filepath,timestamp_utc,distance_mpc,z,pointings
0,0,GW280102a,1856,gammapy_models/GW280102a_catO5_1856_gammapy.fits,GCNs/GW280102a_GCN.json,2028-01-02 00:03:39.819055,140.0,0.031,[]
1,1,GW280103a,110,gammapy_models/GW280103a_catO5_110_gammapy.fits,GCNs/GW280103a_GCN.json,2028-01-03 22:51:35.742546,1480.0,0.280,[]
2,2,GW280104a,1931,gammapy_models/GW280104a_catO5_1931_gammapy.fits,GCNs/GW280104a_GCN.json,2028-01-04 08:31:39.312458,701.0,0.143,[]
3,3,GW280105a,1411,gammapy_models/GW280105a_catO5_1411_gammapy.fits,GCNs/GW280105a_GCN.json,2028-01-05 08:44:05.585539,987.0,0.196,[]
4,4,GW280105b,589,gammapy_models/GW280105b_catO5_589_gammapy.fits,GCNs/GW280105b_GCN.json,2028-01-05 20:46:33.688458,706.0,0.144,[]
...,...,...,...,...,...,...,...,...,...
4609,4609,GW341228a,291,gammapy_models/GW341228a_catO5_291_gammapy.fits,GCNs/GW341228a_GCN.json,2034-12-28 05:21:15.836708,295.0,0.064,[]
4610,4610,GW341228b,1513,gammapy_models/GW341228b_catO5_1513_gammapy.fits,GCNs/GW341228b_GCN.json,2034-12-28 07:22:17.952688,684.0,0.140,[]
4611,4611,GW341228c,104,gammapy_models/GW341228c_catO5_104_gammapy.fits,GCNs/GW341228c_GCN.json,2034-12-28 07:57:55.521803,537.0,0.112,[]
4612,4612,GW341228d,1057,gammapy_models/GW341228d_catO5_1057_gammapy.fits,GCNs/GW341228d_GCN.json,2034-12-28 15:36:18.069099,1350.0,0.258,[]
